# 번호판 인식 모델 파인튜닝 (EasyOCR korean_g2 → 번호판 전용)

**런타임 → 런타임 유형 변경 → T4 GPU** 로 먼저 바꾼다. 안 바꾸면 CPU로 돌아 몇 배 느리다.

## 무엇을 하는가

EasyOCR 한국어 모델(`korean_g2`)을 가져와 **번호판만 읽도록 다시 학습**한다.

밑바닥부터 만들지 않는다. 글자 모양을 알아보는 능력(VGG 특징추출 + 양방향 LSTM)은
이미 학습돼 있으므로 그대로 물려받고, **마지막 판단층만 번호판용으로 바꾼다.**

## 핵심 이점 — 읽을 수 있는 글자를 67자로 줄인다

    korean_g2   약 1,700자 (한글 음절 전체 + 영문 + 기호)
    우리 모델      67자 (숫자 10 + 번호판 한글 40 + 지역명 글자 17)

`3`을 `5`로 잘못 읽는 일은 남지만, **`3`을 `강`으로 읽는 일은 구조적으로 불가능**해진다.
후보가 줄면 그만큼 정확해진다.


## 1. 환경 확인

In [ ]:
!nvidia-smi -L || echo "GPU 없음 — 런타임 유형을 T4 GPU 로 바꿀 것"
import torch
print("torch", torch.__version__, "| CUDA 사용 가능:", torch.cuda.is_available())

In [ ]:
!pip -q install easyocr
# easyocr 를 설치하는 이유는 OCR 을 돌리려는 게 아니라
#   (1) 모델 구조(VGG+BiLSTM) 정의와
#   (2) korean_g2 사전학습 가중치
# 두 가지를 그대로 쓰기 위해서다.

## 2. 학습 데이터 올리기 — 구글 드라이브

데이터가 6만 장(약 750MB)이라 파일 선택 업로드는 자주 끊긴다.
**드라이브에 한 번 올려두면 세션이 끊겨도 다시 안 올려도 된다.**

로컬에서 압축:

```powershell
cd C:\Users\박지원\Desktop\d\omeca-lpr\output\aihub_train
Compress-Archive -Path train,val -DestinationPath ..\aihub_train.zip -Force
```

만들어진 `aihub_train.zip` 을 [drive.google.com](https://drive.google.com) **내 드라이브
최상위**에 드래그해서 올린다. 그다음 아래 셀을 실행한다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!unzip -q -o /content/drive/MyDrive/aihub_train.zip -d /content/data
!ls /content/data
!head -3 /content/data/train/gt.txt
!echo "train $(ls /content/data/train/*.jpg /content/data/train/*.png 2>/dev/null | wc -l) 장"
!echo "val   $(ls /content/data/val/*.jpg /content/data/val/*.png 2>/dev/null | wc -l) 장"

### 2-1. 압축 결과 점검

In [ ]:
# 압축이 제대로 풀렸는지 점검한다.
# Windows 의 Compress-Archive 는 경로 구분자를 역슬래시로 넣는 경우가 있어,
# 리눅스에서 풀면 폴더가 안 생기고 'train\000000.png' 같은 **파일 하나**가 된다.
from pathlib import Path

root = Path('/content/data')
print("최상위:", [p.name for p in root.iterdir()][:10])
for split in ('train', 'val'):
    d = root / split
    if not d.is_dir():
        print(f"  [문제] {d} 폴더가 없다")
        continue
    pngs = list(d.glob('*.png')) + list(d.glob('*.jpg'))
    gt = (d / 'gt.txt')
    n_gt = len(gt.read_text(encoding='utf-8').strip().split('\n')) if gt.exists() else 0
    print(f"  {split}: png {len(pngs)}개 / gt.txt {n_gt}줄  "
          f"{'OK' if len(pngs) == n_gt else '← 개수가 다르다'}")

# 역슬래시 파일명이 생겼다면 여기서 폴더로 되돌린다
odd = [p for p in root.rglob('*') if '\\' in p.name]
if odd:
    print(f"\n역슬래시 파일 {len(odd)}개 발견 → 폴더 구조로 복원한다")
    for p in odd:
        tgt = root / p.name.replace('\\', '/')
        tgt.parent.mkdir(parents=True, exist_ok=True)
        p.rename(tgt)
    print("복원 완료")

## 3. 글자 집합

번호판에 나올 수 있는 글자만 담는다. 이 목록에 없는 글자는 모델이 **출력 자체를 못 한다.**

In [ ]:
CHARS = "0123456789가강거경고광구기나남너노누다대더도두라러로루마머모무바배버보부북사산서세소수아어오우울원인자저전제조종주천충하허호"
print(len(CHARS), "자")

# CTC 는 0번을 [blank] 로 쓴다. 그래서 실제 글자는 1번부터 시작한다.
char2idx = {c: i + 1 for i, c in enumerate(CHARS)}
idx2char = {i + 1: c for i, c in enumerate(CHARS)}
NUM_CLASS = len(CHARS) + 1
print("출력 클래스 수:", NUM_CLASS)

## 4. 데이터셋

**추론할 때와 똑같이 처리해야 한다.** EasyOCR 은 이렇게 한다.

  1. 높이 64px 로 맞추되 가로세로 비를 유지
  2. 가로가 모자라면 **마지막 열을 늘려 채운다** (검은색으로 채우면 그게 가짜 글자처럼 보인다)
  3. 픽셀값을 -1 ~ 1 로 변환

아래는 `easyocr/recognition.py` 의 `NormalizePAD` 를 그대로 옮긴 것이다.

In [ ]:
import math, random
from pathlib import Path
import numpy as np, torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

IMG_H, IMG_W = 64, 300      # 폭 300 → 최대 라벨 길이 30자. 번호판은 최대 9자라 넉넉하다.
MAX_LABEL = 25

class PlateSet(Dataset):
    def __init__(self, root, train=False):
        self.root = Path(root)
        self.items = []
        skipped_char = skipped_missing = 0
        for line in (self.root / "gt.txt").read_text(encoding="utf-8").strip().split("\n"):
            if not line.strip():
                continue
            f, lab = line.split("\t")
            if not all(c in char2idx for c in lab):     # 목록 밖 글자는 학습 불가
                skipped_char += 1
                continue
            if not (self.root / f).exists():            # 업로드 중 빠진 파일
                skipped_missing += 1
                continue
            self.items.append((f, lab))
        self.train = train
        self.to_tensor = T.ToTensor()
        if skipped_char or skipped_missing:
            print(f"  [{self.root.name}] 건너뜀 — 글자 목록 밖 {skipped_char}장 / "
                  f"파일 없음 {skipped_missing}장")

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        f, lab = self.items[i]
        img = Image.open(self.root / f).convert("L")

        if self.train:
            img = self._augment(img)

        w, h = img.size
        ratio = w / float(h)
        rw = min(IMG_W, math.ceil(IMG_H * ratio))
        img = img.resize((rw, IMG_H), Image.BICUBIC)

        t = self.to_tensor(img)
        t.sub_(0.5).div_(0.5)                          # -1 ~ 1
        pad = torch.zeros(1, IMG_H, IMG_W)
        pad[:, :, :rw] = t
        if rw < IMG_W:                                 # 마지막 열로 채우기 (검정 아님)
            pad[:, :, rw:] = t[:, :, rw - 1].unsqueeze(2).expand(1, IMG_H, IMG_W - rw)
        return pad, lab

    @staticmethod
    def _augment(img):
        """실제 배치 조건을 흉내내는 증강.

        1차·2차 학습의 증강은 밝기·노이즈·±3도 회전뿐이었다. 그건 논문
        표준(회전 ±15도, 원근 변형, 블러, 확대축소, shear)에 크게 못 미친다.
        약한 증강은 두 가지를 동시에 놓친다.

          · 과적합을 못 막는다  — 1차에서 loss 는 내려가는데 val 이 멈춘 것이 그 증거다
          · 배치 조건을 못 배운다 — CCTV 는 비스듬하고 흐리고 작다

        아래는 **번호판에 실제로 일어나는 일**만 넣었다. 좌우 반전 같은
        일어날 수 없는 변형은 넣지 않는다.
        """
        import cv2 as _cv
        a = np.array(img)
        h, w = a.shape[:2]

        # ① 원근 왜곡 — 비스듬히 본 번호판. 논문이 가장 강조하는 항목이다.
        if random.random() < 0.5:
            m = 0.10                      # 꼭짓점을 최대 10% 까지 흔든다
            src = np.float32([[0, 0], [w, 0], [w, h], [0, h]])
            dst = src + np.float32([[random.uniform(-m, m) * w,
                                     random.uniform(-m, m) * h] for _ in range(4)])
            a = _cv.warpPerspective(a, _cv.getPerspectiveTransform(src, dst), (w, h),
                                    borderMode=_cv.BORDER_REPLICATE)

        # ② 회전 + 기울임 — 카메라 각도. ±3도는 너무 좁았다.
        if random.random() < 0.6:
            ang = random.uniform(-8, 8)
            shear = random.uniform(-0.10, 0.10)
            M = _cv.getRotationMatrix2D((w / 2, h / 2), ang, 1.0)
            M[0, 1] += shear
            a = _cv.warpAffine(a, M, (w, h), borderMode=_cv.BORDER_REPLICATE)

        # ③ 저해상도 — 멀리서 찍힌 번호판. 줄였다 늘리면 정보가 실제로 사라진다.
        if random.random() < 0.35:
            f = random.uniform(0.35, 0.75)
            small = _cv.resize(a, (max(8, int(w * f)), max(8, int(h * f))),
                               interpolation=_cv.INTER_AREA)
            a = _cv.resize(small, (w, h), interpolation=_cv.INTER_LINEAR)

        # ④ 블러 — 움직이는 차, 초점 흐림
        r = random.random()
        if r < 0.25:                                   # 모션 블러
            k = random.choice([3, 5, 7])
            ker = np.zeros((k, k), np.float32)
            if random.random() < 0.5:
                ker[k // 2, :] = 1.0 / k               # 가로 방향
            else:
                ker[:, k // 2] = 1.0 / k               # 세로 방향
            a = _cv.filter2D(a, -1, ker)
        elif r < 0.45:                                 # 초점 흐림
            a = _cv.GaussianBlur(a, (0, 0), random.uniform(0.6, 1.8))

        # ⑤ 밝기·대비 — 야간, 역광, 그림자
        if random.random() < 0.6:
            a = a.astype(np.float32) * random.uniform(0.6, 1.35) + random.uniform(-35, 35)
            a = np.clip(a, 0, 255).astype(np.uint8)

        # ⑥ 노이즈 + 압축 흔적 — 저조도 센서, JPEG 재압축
        if random.random() < 0.3:
            a = np.clip(a.astype(np.float32) +
                        np.random.normal(0, random.uniform(2, 9), a.shape), 0, 255).astype(np.uint8)
        if random.random() < 0.25:
            q = random.randint(35, 80)
            ok, buf = _cv.imencode('.jpg', a, [int(_cv.IMWRITE_JPEG_QUALITY), q])
            if ok:
                a = _cv.imdecode(buf, _cv.IMREAD_GRAYSCALE)

        return Image.fromarray(a)

def collate(batch):
    imgs = torch.stack([b[0] for b in batch])
    labs = [b[1] for b in batch]
    targets = torch.cat([torch.tensor([char2idx[c] for c in l], dtype=torch.long) for l in labs])
    lengths = torch.tensor([len(l) for l in labs], dtype=torch.long)
    return imgs, targets, lengths, labs


train_ds = PlateSet("/content/data/train", train=True)
val_ds   = PlateSet("/content/data/val",   train=False)
print(f"train {len(train_ds)}장 / val {len(val_ds)}장")

train_dl = DataLoader(train_ds, batch_size=64, shuffle=True,  num_workers=2,
                      collate_fn=collate, drop_last=True, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=64, shuffle=False, num_workers=2,
                      collate_fn=collate, pin_memory=True)

## 5. 모델 — 사전학습 가중치 물려받기

`korean_g2` 는 마지막 층이 **1,700자용**이라 우리 67자와 크기가 다르다. 그래서 보통은
마지막 층을 버리고 처음부터 학습한다.

여기서는 한 걸음 더 간다. **우리 글자가 원래 모델에서 몇 번이었는지 찾아 그 줄만 복사한다.**
`가` 를 판단하던 가중치를 그대로 물려받는 것이다. 버리고 시작하는 것보다 훨씬 빨리 수렴한다.

In [ ]:
import easyocr, os, zipfile, urllib.request
from easyocr.config import recognition_models
from easyocr.model.vgg_model import Model

# --- korean_g2 내려받기 ---------------------------------------------------
cfg = recognition_models['gen2']['korean_g2']
os.makedirs('/root/.EasyOCR/model', exist_ok=True)
pth = '/root/.EasyOCR/model/korean_g2.pth'
if not os.path.exists(pth):
    urllib.request.urlretrieve(cfg['url'], '/content/korean_g2.zip')
    zipfile.ZipFile('/content/korean_g2.zip').extractall('/root/.EasyOCR/model')
print("사전학습 가중치:", pth)

BASE_CHARS = cfg['characters']            # korean_g2 가 아는 글자 (약 1,700자)
print("korean_g2 글자 수:", len(BASE_CHARS))

# --- 모델 만들기 -----------------------------------------------------------
# generation2 규격: input_channel=1, output_channel=256, hidden_size=256
model = Model(input_channel=1, output_channel=256, hidden_size=256, num_class=NUM_CLASS)

sd = torch.load(pth, map_location='cpu', weights_only=False)
sd = {k[7:] if k.startswith('module.') else k: v for k, v in sd.items()}   # 'module.' 제거

pred_w = sd.pop('Prediction.weight')      # [1700+1, 256]
pred_b = sd.pop('Prediction.bias')
missing = model.load_state_dict(sd, strict=False)
print("불러오지 못한 층:", missing.missing_keys)   # Prediction 만 나와야 정상

# --- 마지막 층: 우리 글자에 해당하는 줄만 복사 ------------------------------
with torch.no_grad():
    model.Prediction.weight.zero_(); model.Prediction.bias.zero_()
    model.Prediction.weight[0] = pred_w[0]      # [blank] 는 0번으로 동일
    model.Prediction.bias[0]   = pred_b[0]
    hit = 0
    for c, i in char2idx.items():
        j = BASE_CHARS.find(c)
        if j >= 0:
            model.Prediction.weight[i] = pred_w[j + 1]
            model.Prediction.bias[i]   = pred_b[j + 1]
            hit += 1
print(f"마지막 층 가중치 물려받음: {hit}/{len(CHARS)}자")

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)

## 6. 학습

CTC 손실을 쓴다. 글자가 몇 번째 칸에 있는지 일일이 표시하지 않아도, 순서만 맞으면
알아서 정렬을 찾아 주는 방식이다.

특징추출층은 **낮은 학습률**로 살살 건드리고, 새로 만든 마지막 층은 크게 움직인다.
이미 잘 배운 것을 망가뜨리지 않기 위해서다.

### 1차 학습에서 배운 것

    데이터 2만 장 · 15 에폭  →  val 84.6% 에서 멈춤 (loss 는 계속 하락)

**loss 는 떨어지는데 val 이 안 오르는 것은 데이터가 모자라 외우기 시작했다는
신호다.** AI Hub 공식 벤치마크는 같은 데이터로 99.75% 다. 8만 장 중 2만 장만
쓴 것이 원인으로 보인다 (글자당 상한 600 으로 조여서 '바' 16,126장이 530장이 됐다).

그래서 이번에는 **6만 장 · 35 에폭**으로 올린다. T4 기준 1시간 30분쯤 걸린다.
중간에 끊길 수 있으니 최고 성능은 매 에폭 저장한다.

### 끊김 대비

체크포인트를 **드라이브**(`MyDrive/plate_ckpt/`)에 저장한다. 세션이 끊겨도
남으므로, 다시 들어와 1~5번 셀을 실행하고 이 셀을 다시 돌리면 **끊긴 지점부터
이어서** 학습한다.

학습 중에는 탭을 열어두고 가끔 스크롤하거나 셀을 클릭한다. 무료 Colab 은
조작이 없으면 90분쯤 뒤에 끊는데, 학습 시간(80~90분)과 비슷해서 위험하다.


In [ ]:
import time

EPOCHS = 35
crit = torch.nn.CTCLoss(blank=0, zero_infinity=True)
opt = torch.optim.AdamW([
    {"params": model.FeatureExtraction.parameters(), "lr": 1e-4},
    {"params": model.SequenceModeling.parameters(),  "lr": 3e-4},
    {"params": model.Prediction.parameters(),        "lr": 1e-3},
], weight_decay=1e-4)
sched = torch.optim.lr_scheduler.OneCycleLR(
    opt, max_lr=[3e-4, 1e-3, 3e-3], total_steps=EPOCHS * len(train_dl), pct_start=0.2)
# torch 버전에 따라 API 위치가 다르다
try:
    scaler = torch.amp.GradScaler('cuda', enabled=(device == 'cuda'))
    autocast = lambda: torch.amp.autocast('cuda', enabled=(device == 'cuda'))
except (AttributeError, TypeError):
    scaler = torch.cuda.amp.GradScaler(enabled=(device == 'cuda'))
    autocast = lambda: torch.cuda.amp.autocast(enabled=(device == 'cuda'))


def decode(logits):
    """CTC 결과 → 문자열. 같은 글자 반복과 blank 를 제거한다."""
    out = []
    for seq in logits.argmax(2).cpu().numpy():
        s, prev = [], 0
        for k in seq:
            if k != prev and k != 0:
                s.append(idx2char.get(int(k), ''))
            prev = k
        out.append(''.join(s))
    return out


@torch.no_grad()
def evaluate():
    model.eval()
    ok = tot = 0
    for imgs, _, _, labs in val_dl:
        logits = model(imgs.to(device), None)
        for p, t in zip(decode(logits), labs):
            ok += (p == t); tot += 1
    return ok / max(1, tot)


# **체크포인트는 드라이브에 저장한다.**
#   /content 는 세션이 끊기면 통째로 사라진다. 무료 Colab 은 조작이 없으면
#   90분쯤 뒤 끊는데 학습 시간과 비슷해서, 다 돌고 나서 날리기 십상이다.
#   드라이브에 두면 끊겨도 남는다.
import os
CKPT_DIR = '/content/drive/MyDrive/plate_ckpt'
os.makedirs(CKPT_DIR, exist_ok=True)
CKPT = f'{CKPT_DIR}/plate_best.pth'

# 이어하기 — 끊겼다가 다시 왔으면 여기서 이어받는다
start_ep, best = 1, 0.0
if os.path.exists(CKPT):
    ck = torch.load(CKPT, map_location=device)
    if isinstance(ck, dict) and 'model' in ck:
        model.load_state_dict(ck['model']); best = ck.get('acc', 0.0)
        start_ep = ck.get('epoch', 0) + 1
        print(f"이어하기: {start_ep-1} 에폭까지 완료, 최고 val {best:.2%}")

for ep in range(start_ep, EPOCHS + 1):
    model.train(); t0 = time.time(); run = 0.0
    for i, (imgs, targets, lengths, _) in enumerate(train_dl):
        imgs = imgs.to(device, non_blocking=True)
        opt.zero_grad(set_to_none=True)
        with autocast():
            logits = model(imgs, None)                    # [B, T, C]
            logp = logits.log_softmax(2).permute(1, 0, 2) # CTC 는 [T, B, C] 를 받는다
            input_len = torch.full((imgs.size(0),), logp.size(0), dtype=torch.long)
            loss = crit(logp.float(), targets, input_len, lengths)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        scaler.step(opt); scaler.update(); sched.step()
        run += loss.item()
        if (i + 1) % 50 == 0:
            print(f"  ep{ep} {i+1}/{len(train_dl)} loss {run/(i+1):.4f}", end="\r")

    acc = evaluate()
    mark = ""
    if acc > best:
        best = acc
        torch.save({'model': model.state_dict(), 'acc': acc, 'epoch': ep}, CKPT)
        mark = "  ← 저장(드라이브)"
    print(f"epoch {ep:2d}  loss {run/len(train_dl):.4f}  "
          f"val 완전일치 {acc:.2%}  ({time.time()-t0:.0f}초){mark}")

print(f"\n최고 val 정확도 {best:.2%}")

## 7. 틀린 것 살펴보기

숫자만 보지 말고 무엇을 틀리는지 확인한다. 특정 글자에서만 틀리면 데이터가 부족한 것이고,
전반적으로 틀리면 학습이 덜 된 것이다.

In [ ]:
model.load_state_dict(torch.load(CKPT, map_location=device)['model'])
model.eval()

from collections import Counter
wrong, cnt = [], Counter()
with torch.no_grad():
    for imgs, _, _, labs in val_dl:
        for p, t in zip(decode(model(imgs.to(device), None)), labs):
            if p != t:
                wrong.append((t, p))
                if len(t) == len(p):
                    for a, b in zip(t, p):
                        if a != b:
                            cnt[f"{a}→{b}"] += 1

print(f"틀린 것 {len(wrong)} / {len(val_ds)}  ({len(wrong)/len(val_ds):.1%})\n")
print("자주 틀리는 글자쌍:")
for k, v in cnt.most_common(15):
    print(f"  {k}  {v}회")
print("\n예시:")
for t, p in wrong[:25]:
    print(f"  정답 {t:12} → 예측 {p}")

## 8. EasyOCR 에 끼울 수 있게 내보내기

EasyOCR 은 사용자 모델을 쓸 때 파일 세 개를 본다.

    ~/.EasyOCR/model/plate.pth          가중치
    ~/.EasyOCR/user_network/plate.py    모델 구조
    ~/.EasyOCR/user_network/plate.yaml  글자 목록·입력 크기

세 개를 만들어 zip 으로 내려받는다.

In [ ]:
import os, json, shutil
out = '/content/plate_model'
os.makedirs(out, exist_ok=True)

# 1) 가중치 — EasyOCR 은 'module.' 접두사를 붙인 형태를 기대한다
sd = torch.load(CKPT, map_location='cpu')['model']
torch.save({('module.' + k): v for k, v in sd.items()}, f'{out}/plate.pth')

# 2) 모델 구조 (easyocr.model.vgg_model 과 동일)
open(f'{out}/plate.py', 'w', encoding='utf-8').write("""import torch.nn as nn
from easyocr.model.modules import VGG_FeatureExtractor, BidirectionalLSTM

class Model(nn.Module):
    def __init__(self, input_channel, output_channel, hidden_size, num_class):
        super(Model, self).__init__()
        self.FeatureExtraction = VGG_FeatureExtractor(input_channel, output_channel)
        self.FeatureExtraction_output = output_channel
        self.AdaptiveAvgPool = nn.AdaptiveAvgPool2d((None, 1))
        self.SequenceModeling = nn.Sequential(
            BidirectionalLSTM(self.FeatureExtraction_output, hidden_size, hidden_size),
            BidirectionalLSTM(hidden_size, hidden_size, hidden_size))
        self.SequenceModeling_output = hidden_size
        self.Prediction = nn.Linear(self.SequenceModeling_output, num_class)

    def forward(self, input, text):
        visual_feature = self.FeatureExtraction(input)
        visual_feature = self.AdaptiveAvgPool(visual_feature.permute(0, 3, 1, 2))
        visual_feature = visual_feature.squeeze(3)
        contextual_feature = self.SequenceModeling(visual_feature)
        return self.Prediction(contextual_feature.contiguous())
""")

# 3) 설정
yaml_txt = f"""network_params:
  input_channel: 1
  output_channel: 256
  hidden_size: 256
imgH: 64
lang_list:
  - 'ko'
character_list: {CHARS}
"""
open(f'{out}/plate.yaml', 'w', encoding='utf-8').write(yaml_txt)

shutil.make_archive('/content/plate_model', 'zip', out)
print("완성:")
!ls -la /content/plate_model

from google.colab import files
files.download('/content/plate_model.zip')

## 9. 내 PC 에 설치

내려받은 `plate_model.zip` 을 풀고 아래 위치에 넣는다.

    C:\Users\박지원\.EasyOCR\model\plate.pth
    C:\Users\박지원\.EasyOCR\user_network\plate.py
    C:\Users\박지원\.EasyOCR\user_network\plate.yaml

그다음 `omeca-lpr` 폴더에서 실측한다.

```powershell
python bench_lpr.py --save-fail
```

**기존 78% 와 비교해서 올랐는지 확인한다.** 안 올랐으면 되돌리면 된다 —
`plate.pth` 만 지우면 원래 모델로 돌아간다.